# Reference DreamerV3 on Pendulum-v1 (matched hyperparameters)

A/B test against our implementation. Runs the standard **NM512/dreamerv3-torch**
on **Pendulum-v1 (state-based obs)** with hyperparameters matched to ours, logging
`eval_return` and saving parity videos (real-env + imagination).

**Question:** does the standard library also diverge under our hyperparameters, or
does it learn a stable swing-up? If it converges stably, the divergence is a bug in
*our* code, not the algorithm/task.

Run cells top to bottom. **You must push the `benchmarks/` folder to your repo first.**
Set the GPU runtime: *Runtime → Change runtime type → GPU (T4/L4/A100)*.

## 1. Clone your repo

In [ ]:
# Clone (public repo). For a private repo, use a token:
#   !git clone https://<TOKEN>@github.com/aritraban21/Dreamerv3.git
!rm -rf Dreamerv3
!git clone https://github.com/aritraban21/Dreamerv3.git
%cd Dreamerv3/benchmarks/dreamerv3-torch
!ls

## 2. Install lean deps

Only what the **state-based** Pendulum run needs (no mujoco/dm_control/crafter).
`gym==0.22.0` requires `numpy<1.24`, so numpy is downgraded — **you must restart the
runtime once after this cell** (*Runtime → Restart session*), then continue at step 3.

In [ ]:
!pip -q install -r requirements-pendulum.txt
print('\n\n=== Deps installed. NOW: Runtime -> Restart session, then run from step 3. ===')

## 3. (After restart) setup + GPU check

In [ ]:
import os
%cd /content/Dreamerv3/benchmarks/dreamerv3-torch
# headless rendering for pygame (gym import) + matplotlib (video frames)
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['MPLBACKEND'] = 'Agg'
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', np.__version__)
assert np.__version__.startswith('1.23'), 'Restart the runtime after step 2 so numpy 1.23.5 is active.'

## 4. Quick smoke (~1–2 min) — confirm it runs on this Colab

Tiny model, few steps, CPU-fine. Expect `eval_return` printed and two mp4s written.

In [ ]:
!python -u dreamer.py --configs pendulum --logdir ./logdir/smoke \
  --device cuda:0 --compile False --steps 130 --prefill 60 --pretrain 5 \
  --eval_every 120 --eval_episode_num 1 --batch_size 4 --batch_length 16 \
  --dyn_deter 64 --dyn_hidden 64 --units 64 --dyn_stoch 8 --dyn_discrete 8 \
  --imag_horizon 5 --render_video True --imag_video_horizon 8
!echo '--- videos ---'; ls -la videos/pendulum 2>/dev/null

## 5. Full run — matched hyperparameters, 200k steps

This is the real comparison. Same env + hyperparameters as our runs (the `pendulum`
config block sets `dyn_deter=4096`, `dyn_stoch=32×32`, `units=1024`, `imag_horizon=16`,
`batch=16×64`, `train_ratio=512`, `discount=0.997`, actor entropy `3e-4`, …).

`--render_video True` saves a real-env + imagination mp4 at every eval. Output is
teed to `run.log` (like our `logs.txt`). This takes **hours** on a T4 — keep the tab
alive. `--compile False` avoids a long torch.compile warmup (does not change results).

In [ ]:
!python -u dreamer.py --configs pendulum --logdir ./logdir/pendulum_ref \
  --device cuda:0 --compile False --seed 0 --render_video True 2>&1 | tee run.log

## 6. Plot the eval-return curve

Reads `eval_return` from `metrics.jsonl`. To overlay OUR run, set `OUR_LOG` to an
uploaded copy of one of our `Pendulum-v1_*.log` files (it grabs the
`[step N] eval return = X` lines).

In [ ]:
import json, re, pathlib
import matplotlib.pyplot as plt

def ref_curve(metrics_path):
    steps, rets = [], []
    for line in pathlib.Path(metrics_path).read_text().splitlines():
        try: d = json.loads(line)
        except Exception: continue
        if 'eval_return' in d:
            steps.append(d.get('step', len(steps))); rets.append(d['eval_return'])
    return steps, rets

def our_curve(log_path):
    steps, rets = [], []
    for line in pathlib.Path(log_path).read_text().splitlines():
        m = re.search(r'\[step (\d+)\] eval return = (-?[\d.]+)', line)
        if m: steps.append(int(m.group(1))); rets.append(float(m.group(2)))
    return steps, rets

rs, rr = ref_curve('logdir/pendulum_ref/metrics.jsonl')
plt.figure(figsize=(9,5))
plt.plot(rs, rr, '-o', ms=3, label='reference dreamerv3-torch')

OUR_LOG = ''  # e.g. '/content/Pendulum-v1_20260901-205555(2).log' if you upload ours
if OUR_LOG and pathlib.Path(OUR_LOG).exists():
    os_, or_ = our_curve(OUR_LOG)
    plt.plot(os_, or_, '-s', ms=3, label='ours')

plt.axhline(-200, ls=':', c='gray', label='~solved band (-150..-250)')
plt.xlabel('env step'); plt.ylabel('eval return'); plt.legend(); plt.grid(alpha=.3)
plt.title('Pendulum-v1: reference vs ours (matched hyperparameters)')
plt.show()
print('reference eval_returns:', [round(x,1) for x in rr])

## 7. Watch a video (real env + imagination)

In [ ]:
import glob
from IPython.display import Video, display
envs = sorted(glob.glob('videos/pendulum/env_step_*.mp4'))
imgs = sorted(glob.glob('videos/pendulum/imag_step_*.mp4'))
if envs: print('real env:', envs[-1]); display(Video(envs[-1], embed=True, width=320))
if imgs: print('imagination:', imgs[-1]); display(Video(imgs[-1], embed=True, width=320))